# Define Forecasting Models

This notebook defines the XGBoost forecasting models.

One model predicts solar PV generation. The other predicts electricity consumption. The actual training and evaluation happen in the analysis notebooks.

## Why This Notebook Matters

Keeping the model definitions in one notebook makes the project easier to maintain. The same model classes can be reused for one-day forecasts and longer forecast horizons.

In [7]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project
Notebook folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project/Notebooks


In [8]:
# Import packages and features.
os.chdir(notebook_folder)

import import_ipynb
import xgboost as xgb
import hashlib
import numpy as np
import pandas as pd

from sklearn.metrics import r2_score, mean_absolute_error
from Feature_Loader import PVFeatures, ConFeatures, HolidayFeatures

os.chdir(project_folder)

print("Current folder:", os.getcwd())

Current folder: /Users/mac/Library/CloudStorage/OneDrive-UniversityofKeele/Dissertation/send-ev-project


In [9]:
# Define metrics.
class EvaluationMetrics:

    @staticmethod
    def calculate_and_print(y_real, y_pred):
        y_hourly_actual = y_real.resample("1h").mean()
        y_hourly_preds = y_pred.resample("1h").mean()

        y_daily_actual = y_real.resample("1D").mean()
        y_daily_preds = y_pred.resample("1D").mean()

        r2 = r2_score(y_real, y_pred)
        mae = mean_absolute_error(y_real, y_pred)
        mbe = np.mean(y_real - y_pred)

        hourly_r2 = r2_score(y_hourly_actual, y_hourly_preds)
        hourly_mae = mean_absolute_error(y_hourly_actual, y_hourly_preds)

        daily_r2 = r2_score(y_daily_actual, y_daily_preds)
        daily_mae = mean_absolute_error(y_daily_actual, y_daily_preds)

        print(f"5-Minute R2 Score: {r2:.4f} | MAE: {mae:.2f} kW | MBE: {mbe:.2f} kW")
        print(f"Hourly R2 Score:   {hourly_r2:.4f} | MAE: {hourly_mae:.2f} kW")
        print(f"Daily R2 Score:    {daily_r2:.4f} | MAE: {daily_mae:.2f} kW")

In [10]:
# Define solar model.
class PVModel:

    def __init__(self, features, model_dir="Models/"):
        self.features = features

        self.model_dir = model_dir

        self.target_col = "power-gen-pv-ave"

        os.makedirs(self.model_dir, exist_ok=True)

    def _get_model_path(self):
        feat_str = "_".join(sorted(self.features))
        feat_hash = hashlib.md5(feat_str.encode()).hexdigest()[:8]

        return os.path.join(self.model_dir, f"pv_model_feats_{feat_hash}.json")

    def train_and_test(self, deop, solcast, deop_test, solcast_test, days_ahead=1):
        X_train_full = PVFeatures.create_pv_features(
            deop,
            solcast,
            days_ahead=1,
        )

        X_test_full = PVFeatures.create_pv_features(
            deop_test,
            solcast_test,
            days_ahead=days_ahead,
        )

        X_train = X_train_full[self.features]
        X_test = X_test_full[self.features]

        y_train = deop.loc[X_train.index, self.target_col]
        y_test = deop_test.loc[X_test.index, self.target_col]

        split_idx = int(len(X_train) * 0.90)

        X_tr = X_train.iloc[:split_idx]
        y_tr = y_train.iloc[:split_idx]

        X_val = X_train.iloc[split_idx:]
        y_val = y_train.iloc[split_idx:]

        model_path = self._get_model_path()

        model = xgb.XGBRegressor(
            objective="reg:tweedie",
            tweedie_variance_power=1.5,
            n_estimators=1500,
            learning_rate=0.01,
            max_depth=6,
            reg_lambda=0.5,
            subsample=0.9,
            colsample_bytree=0.8,
            random_state=13,
            tree_method="hist",
            early_stopping_rounds=150,
        )

        if os.path.exists(model_path):
            print(f"Loading PV model from {model_path}")
            model.load_model(model_path)
        else:
            print("Training new PV model")
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            model.save_model(model_path)

        raw_preds = model.predict(X_test)

        mask_daylight = (X_test.index.hour >= 4) & (X_test.index.hour < 21)

        final_preds = pd.Series(
            np.where(mask_daylight, np.maximum(0, raw_preds), 0),
            index=X_test.index,
        )

        EvaluationMetrics.calculate_and_print(y_test, final_preds)

        return final_preds, model

In [11]:
# Define consumption model.
class ConsumptionModel:

    def __init__(self, features, model_dir="Models/"):
        self.features = features

        self.model_dir = model_dir

        self.target_col = "power-con-ave"

        os.makedirs(self.model_dir, exist_ok=True)

    def _get_model_path(self):
        feat_str = "_".join(sorted(self.features))
        feat_hash = hashlib.md5(feat_str.encode()).hexdigest()[:8]

        return os.path.join(self.model_dir, f"con_model_feats_{feat_hash}.json")

    def train_and_test(self, deop, solcast, deop_test, solcast_test, days_ahead=1):
        X_train_full = ConFeatures.create_consumption_features(
            deop,
            solcast,
            days_ahead=1,
        )[self.features]

        X_test_full = ConFeatures.create_consumption_features(
            deop_test,
            solcast_test,
            days_ahead=days_ahead,
        )[self.features]

        y_train_residual = (
            deop.loc[X_train_full.index, self.target_col]
            - X_train_full["rolling_7d_baseline"]
        )

        y_test_real = deop_test.loc[X_test_full.index, self.target_col]

        split_idx = int(len(X_train_full) * 0.90)

        X_tr = X_train_full.iloc[:split_idx]
        y_tr = y_train_residual.iloc[:split_idx]

        X_val = X_train_full.iloc[split_idx:]
        y_val = y_train_residual.iloc[split_idx:]

        model_path = self._get_model_path()

        model = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=1500,
            learning_rate=0.01,
            max_depth=5,
            min_child_weight=5,
            reg_lambda=5.0,
            reg_alpha=1.0,
            subsample=0.9,
            colsample_bytree=0.7,
            random_state=13,
            tree_method="exact",
            early_stopping_rounds=150,
        )

        if os.path.exists(model_path):
            print(f"Loading consumption model from {model_path}")
            model.load_model(model_path)
        else:
            print("Training new consumption model")
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            model.save_model(model_path)

        predicted_residuals = model.predict(X_test_full)

        raw_preds = predicted_residuals + X_test_full["rolling_7d_baseline"]

        final_preds = pd.Series(
            np.maximum(0, raw_preds),
            index=X_test_full.index,
        )

        EvaluationMetrics.calculate_and_print(y_test_real, final_preds)

        return final_preds, model

In [12]:
# Check model classes.
print(PVModel)
print(ConsumptionModel)
print(EvaluationMetrics)

<class '__main__.PVModel'>
<class '__main__.ConsumptionModel'>
<class '__main__.EvaluationMetrics'>


## What This Notebook Achieved

This notebook defines the forecasting models. The solar model predicts PV generation directly, while the consumption model predicts demand relative to a rolling 7-day baseline.

In simple terms, this notebook answers three questions: what is being done, why it matters, and what it produces for the next step.